# Notebook 04 — Learned Wavelet Experiments (FordA)

Classificação binária com modelos DL usando wavelets aprendidas end-to-end (LearnedWaveletDWT1D_QMF).
- **LearnedWavelet + CNN**
- **LearnedWavelet + LSTM**
- **LearnedWavelet + CNN-LSTM**
- **LearnedWavelet + Transformer**

**Pipeline:** Sinal Raw (500,1) → LearnedWaveletDWT1D_QMF → Backbone → Sigmoid → Avaliação

In [ ]:
import os, sys
sys.path.append(".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "..", ".."))
from config.experiment_config import (
    DATA_DIR, RESULTS_DIR, MODELS_DIR,
    DL_TRAINING_CONFIG, LEARNED_WAVELET_CONFIG, LEARNED_WAVELET_MODELS_CONFIG,
    generate_learned_wavelet_grid, SEED, GPU_ID, EPOCHS_OVERRIDE, MAX_GRID_CONFIGS,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import random
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
# Limitar a 1 GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices([gpus[0]], 'GPU')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

from src.models import (
    create_learned_wavelet_cnn_model, create_learned_wavelet_lstm_model,
    create_learned_wavelet_cnn_lstm_model, create_learned_wavelet_transformer_model,
    get_callbacks, get_distribute_strategy,
)
from src.evaluation import ClassificationEvaluator, ResultsManager
from src.visualization import ExperimentVisualizer

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

strategy = get_distribute_strategy()

DL_LW_DIR = RESULTS_DIR / "learned_wavelet_experiments"
DL_LW_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nLearnedWaveletDWT1D_QMF carregado")
print(f"SEED={SEED}")
if GPU_ID: print(f"GPU selecionada: {GPU_ID}")
if EPOCHS_OVERRIDE: print(f"EPOCHS_OVERRIDE={EPOCHS_OVERRIDE}")
if MAX_GRID_CONFIGS: print(f"MAX_GRID_CONFIGS={MAX_GRID_CONFIGS}")

## 1. Carregar Dados

In [ ]:
X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

X_train = X_train[..., np.newaxis]  # (N, 500, 1)
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"Dados Carregados (Raw + Canal):")
print(f"  Train: {X_train.shape}, y: {y_train.shape} (classe 1: {y_train.mean():.2%})")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")

input_shape = X_train.shape[1:]
print(f"\nInput shape: {input_shape}")

## 2. Configuração das Learned Wavelets

In [ ]:
wavelet_config = LEARNED_WAVELET_CONFIG.copy()
print("Configuração LearnedWaveletDWT1D_QMF:")
for k, v in wavelet_config.items():
    print(f"  {k}: {v}")

results_manager = ResultsManager(DL_LW_DIR)
evaluator = ClassificationEvaluator()
viz = ExperimentVisualizer()

training_config = DL_TRAINING_CONFIG.copy()

all_results = {}
all_histories = {}
all_grid_results = []
best_models = {}  # guardar modelos Keras para análise de filtros

## 3. Definição dos Modelos

Modelos criados pelo factory em `src/models.py`:
- `create_learned_wavelet_cnn_model`
- `create_learned_wavelet_lstm_model`
- `create_learned_wavelet_cnn_lstm_model`
- `create_learned_wavelet_transformer_model`

Parâmetros base em `LEARNED_WAVELET_MODELS_CONFIG` e grid em `LEARNED_WAVELET_GRID_AXES`.

In [ ]:
# Wrapping model builders para interface uniforme (params=)
MODEL_BUILDERS = {
    "CNN": lambda inp, wavelet_config=None, params=None: create_learned_wavelet_cnn_model(inp, wavelet_config=wavelet_config, cnn_params=params),
    "LSTM": lambda inp, wavelet_config=None, params=None: create_learned_wavelet_lstm_model(inp, wavelet_config=wavelet_config, lstm_params=params),
    "CNN_LSTM": lambda inp, wavelet_config=None, params=None: create_learned_wavelet_cnn_lstm_model(inp, wavelet_config=wavelet_config, cnn_lstm_params=params),
    "Transformer": lambda inp, wavelet_config=None, params=None: create_learned_wavelet_transformer_model(inp, wavelet_config=wavelet_config, transformer_params=params),
}

## 4. Experimento 1: LearnedWavelet + CNN

In [ ]:
print("=" * 70)
print("Grid Search: LearnedWavelet + CNN")
print("=" * 70)

arch = "CNN"
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"LW_CNN_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](
            input_shape, wavelet_config=wavelet_config, params=params
        )

    model_path = str(MODELS_DIR / f"lw_cnn_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        patience_lr=training_config["reduce_lr_patience"],
        min_lr=training_config["min_lr"],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "LW_CNN", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["LW_CNN"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["LW_CNN"] = history.history
        best_models["LW_CNN"] = model
        model.save(str(MODELS_DIR / "lw_cnn_best.keras"))

    results_manager.log_experiment("Learned_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor CNN: {best_key} — Acc={best_acc:.4f}")


## 5. Experimento 2: LearnedWavelet + LSTM

In [ ]:
print("=" * 70)
print("Grid Search: LearnedWavelet + LSTM")
print("=" * 70)

arch = "LSTM"
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"LW_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](
            input_shape, wavelet_config=wavelet_config, params=params
        )

    model_path = str(MODELS_DIR / f"lw_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        patience_lr=training_config["reduce_lr_patience"],
        min_lr=training_config["min_lr"],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "LW_LSTM", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["LW_LSTM"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["LW_LSTM"] = history.history
        best_models["LW_LSTM"] = model
        model.save(str(MODELS_DIR / "lw_lstm_best.keras"))

    results_manager.log_experiment("Learned_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor LSTM: {best_key} — Acc={best_acc:.4f}")


## 6. Experimento 3: LearnedWavelet + CNN_LSTM

In [ ]:
print("=" * 70)
print("Grid Search: LearnedWavelet + CNN_LSTM")
print("=" * 70)

arch = "CNN_LSTM"
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"LW_CNN_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](
            input_shape, wavelet_config=wavelet_config, params=params
        )

    model_path = str(MODELS_DIR / f"lw_cnn_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        patience_lr=training_config["reduce_lr_patience"],
        min_lr=training_config["min_lr"],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "LW_CNN_LSTM", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["LW_CNN_LSTM"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["LW_CNN_LSTM"] = history.history
        best_models["LW_CNN_LSTM"] = model
        model.save(str(MODELS_DIR / "lw_cnn_lstm_best.keras"))

    results_manager.log_experiment("Learned_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor CNN_LSTM: {best_key} — Acc={best_acc:.4f}")


## 7. Experimento 4: LearnedWavelet + Transformer

In [ ]:
print("=" * 70)
print("Grid Search: LearnedWavelet + Transformer")
print("=" * 70)

arch = "Transformer"
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"LW_Transformer_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](
            input_shape, wavelet_config=wavelet_config, params=params
        )

    model_path = str(MODELS_DIR / f"lw_transformer_g{gi}.keras")
    # Transformer usa WarmupSchedule, incompatível com ReduceLROnPlateau
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        use_reduce_lr=False,
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "LW_Transformer", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["LW_Transformer"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["LW_Transformer"] = history.history
        best_models["LW_Transformer"] = model
        model.save(str(MODELS_DIR / "lw_transformer_best.keras"))

    results_manager.log_experiment("Learned_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor Transformer: {best_key} — Acc={best_acc:.4f}")

## 8. Comparação dos Resultados

In [ ]:
comp_rows = []
for name, info in all_results.items():
    row = {"Model": name, **info["metrics"],
           "Params": info["params"], "Time (s)": round(info["time"], 1),
           "Epochs": info["epochs"]}
    comp_rows.append(row)

comparison_df = pd.DataFrame(comp_rows).sort_values("accuracy", ascending=False)
comparison_df = comparison_df.set_index("Model")
print("\n" + "=" * 80)
print("COMPARAÇÃO — Learned Wavelet Models")
print("=" * 80)
print(comparison_df.to_string())

comparison_df.to_csv(DL_LW_DIR / "comparison_dl_learned_wavelet.csv")
pd.DataFrame(all_grid_results).to_csv(DL_LW_DIR / "all_grid_results_dl_learned_wavelet.csv", index=False)
print(f"\nResultados salvos em {DL_LW_DIR}")

In [ ]:
metrics_to_plot = ["accuracy", "f1", "auc_roc"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(6 * len(metrics_to_plot), 5))
model_names = comparison_df.index.tolist()
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(model_names)))
for ax, metric in zip(axes, metrics_to_plot):
    values = comparison_df[metric].values
    bars = ax.barh(range(len(model_names)), values, color=colors)
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names)
    ax.set_xlabel(metric.upper())
    ax.set_title(metric.upper())
    ax.invert_yaxis()
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
fig.suptitle("Comparação Learned Wavelet — FordA", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DL_LW_DIR / "comparison_chart_learned_wavelet.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Evolução do Treinamento

In [ ]:
n_models = len(all_histories)
if n_models > 0:
    fig, axes = plt.subplots(n_models, 2, figsize=(14, 4 * n_models))
    if n_models == 1:
        axes = axes.reshape(1, -1)
    for i, (name, hist) in enumerate(all_histories.items()):
        axes[i, 0].plot(hist["loss"], label="Train")
        axes[i, 0].plot(hist["val_loss"], label="Val")
        axes[i, 0].set_title(f"{name} — Loss")
        axes[i, 0].set_xlabel("Epoch")
        axes[i, 0].legend()
        acc_key = "accuracy" if "accuracy" in hist else "acc"
        val_acc_key = "val_accuracy" if "val_accuracy" in hist else "val_acc"
        if acc_key in hist:
            axes[i, 1].plot(hist[acc_key], label="Train")
            axes[i, 1].plot(hist[val_acc_key], label="Val")
            axes[i, 1].set_title(f"{name} — Accuracy")
            axes[i, 1].set_xlabel("Epoch")
            axes[i, 1].legend()
    plt.tight_layout()
    plt.savefig(DL_LW_DIR / "training_evolution_learned_wavelet.png", dpi=150, bbox_inches="tight")
    plt.show()

## 10. Confusion Matrices e ROC

In [ ]:
from sklearn.metrics import roc_curve, auc

for name, info in all_results.items():
    viz.plot_confusion_matrix(
        y_test, info["y_pred"],
        model_name=name,
        save_path=DL_LW_DIR / f"confusion_matrix_{name}.png",
    )
    plt.show()

roc_data = {}
for name, info in all_results.items():
    if "y_prob" in info:
        fpr, tpr, _ = roc_curve(y_test, info["y_prob"])
        auc_val = auc(fpr, tpr)
        roc_data[name] = (fpr, tpr, auc_val)
if roc_data:
    viz.plot_multi_roc(
        roc_data,
        save_path=DL_LW_DIR / "roc_curves_learned_wavelet.png",
    )
    plt.show()

## 11. Visualização dos Filtros Aprendidos (Preview)

In [ ]:
# Preview rápido dos filtros aprendidos (análise completa no notebook 06)
for name, model in best_models.items():
    # Procurar camada learned wavelet
    lw_layers = [l for l in model.layers if "learned_wavelet" in l.name.lower()
                 or "dwt" in l.name.lower()]
    if not lw_layers:
        print(f"{name}: camada learned wavelet não encontrada")
        continue

    lw = lw_layers[0]
    weights = lw.get_weights()
    if weights:
        fig, axes = plt.subplots(1, len(weights), figsize=(6 * len(weights), 4))
        if len(weights) == 1:
            axes = [axes]
        for ax, (w, wname) in zip(axes, zip(weights, [f"W{i}" for i in range(len(weights))])):
            ax.plot(w.flatten())
            ax.set_title(f"{name} — {wname} (shape={w.shape})")
            ax.grid(True)
        plt.tight_layout()
        plt.savefig(DL_LW_DIR / f"filters_preview_{name}.png", dpi=150, bbox_inches="tight")
        plt.show()

## 12. Resumo

In [ ]:
print("=" * 60)
print("RESUMO — Learned Wavelet Experiments (FordA)")
print("=" * 60)
print(f"Configuração wavelet aprendida:")
for k, v in wavelet_config.items():
    print(f"  {k}: {v}")
print(f"\nArquiteturas avaliadas: {len(all_results)}")
print(f"Total de configurações grid: {len(all_grid_results)}")
if all_results:
    best_name = comparison_df.index[0]
    print(f"Melhor modelo: {best_name}")
    print(f"  Accuracy: {comparison_df.loc[best_name, 'accuracy']:.4f}")
    print(f"  F1:       {comparison_df.loc[best_name, 'f1']:.4f}")
    print(f"  AUC-ROC:  {comparison_df.loc[best_name, 'auc_roc']:.4f}")
print(f"Resultados em: {DL_LW_DIR}")